In [1]:
import numpy as np, pandas as pd, sklearn
print("Setup OK", pd.__version__, sklearn.__version__)

Setup OK 2.3.3 1.7.2


# Lab 3: Ticket Escalation Classification

In this lab I build and compare classifiers that predict whether a service ticket gets escalated. The goal isn't just a high score. I want to pick metrics and a decision threshold that reflect what mistakes actually cost.

## 1. Setup and loading the data

First I import all the libraries I'll need for the whole lab, so everything is in one place at the top.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay
)

I load the CSV and take a first look: how many rows and columns there are, what the first few tickets look like, and the data types of each column.

In [3]:
df = pd.read_csv("classification_escalation.csv")
print("Shape (rows, columns):", df.shape)
display(df.head())
print()
df.info()

Shape (rows, columns): (900, 6)


,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,9,7.7,low,web,21.6,0
1,2,6.1,high,web,21.8,0
2,1,2.5,medium,phone,28.8,0
3,7,5.3,low,email,52.5,0
4,4,7.1,medium,email,14.7,0



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   complexity_score        900 non-null    int64  
 1   team_experience_years   869 non-null    float64
 2   priority                882 non-null    object 
 3   channel                 900 non-null    object 
 4   customer_tenure_months  878 non-null    float64
 5   escalated               900 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 42.3+ KB


The lab says some predictors have missing values on purpose. Here I count them per column, and also show them as a percentage so it's clear how big the problem is. I'll handle them later inside the preprocessing pipeline.

In [4]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(1)
})
display(missing)

,missing_count,missing_percent
complexity_score,0,0.0
team_experience_years,31,3.4
priority,18,2.0
channel,0,0.0
customer_tenure_months,22,2.4
escalated,0,0.0


Next I check how common escalations are. If one class is much rarer than the other, plain accuracy can look good while the model misses most escalated tickets.

In [5]:
print("Proportion of each class:")
print(df["escalated"].value_counts(normalize=True).rename("proportion"))
print()
print("Counts:")
print(df["escalated"].value_counts().rename("count"))

Proportion of each class:
escalated
0    0.793333
1    0.206667
Name: proportion, dtype: float64

Counts:
escalated
0    714
1    186
Name: count, dtype: int64


A quick statistical summary of the numeric columns, plus the categories in the text columns, to see ranges and spot anything odd.

In [6]:
display(df.describe())
for col in ["priority", "channel"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

,complexity_score,team_experience_years,customer_tenure_months,escalated
count,900.000000,869.000000,878.000000,900.000000
mean,5.433333,4.443268,22.553075,0.206667
std,2.834909,2.137840,14.888236,0.405139
min,1.000000,0.000000,0.900000,0.000000
25%,3.000000,2.900000,11.400000,0.000000
50%,5.000000,4.400000,19.550000,0.000000
75%,8.000000,5.900000,30.150000,0.000000
max,10.000000,11.800000,97.400000,1.000000



priority:
priority
low       434
medium    306
high      142
NaN        18
Name: count, dtype: int64

channel:
channel
email    424
web      304
phone    172
Name: count, dtype: int64


**What I found:** The dataset has 900 tickets and 5 predictors. Only three columns have missing values (between 2% and 3.4%), so imputation is enough and no rows need dropping. About 20.7% of tickets are escalated, so the classes are imbalanced. A model that always predicts "not escalated" would already be 79.3% accurate, which is why I'll look at precision, recall and F1 and not accuracy alone.

## 2. Problem framing

This is a **classification** problem because the thing I'm predicting, `escalated`, is a category (yes or no), not a number on a scale. Specifically it's binary classification with two outcomes.

The **positive class is 1 = escalated**. I chose this because escalations are the rare and costly event the support team actually wants to catch. So when I talk about recall later, I mean "of all the tickets that really were escalated, how many did the model find?"

## 3. Train/test split

First I split the table into the predictors (X) and the target (y). Then I hold out 25% of the tickets as a test set that the models never see during training. I use `stratify=y` so both sets keep the same share of escalated tickets (about 20.7%), and `random_state=42` so the split is the same every time I run it.

In [7]:
X = df.drop(columns="escalated")
y = df["escalated"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print()
print("Escalation rate in each set:")
print("full data:", round(y.mean(), 4))
print("train    :", round(y_train.mean(), 4))
print("test     :", round(y_test.mean(), 4))

Train: (675, 5)  Test: (225, 5)

Escalation rate in each set:
full data: 0.2067
train    : 0.2074
test     : 0.2044


I split the data *before* filling in missing values or scaling anything. If I computed medians or averages on the whole dataset first, information from the test set would leak into training and make my results look better than they really are. Doing the cleaning inside a pipeline fitted only on the training data avoids that.

## 4. Preprocessing

The numeric and categorical columns need different treatment, so I list them separately first.

In [8]:
numeric_features = ["complexity_score", "team_experience_years", "customer_tenure_months"]
categorical_features = ["priority", "channel"]

print("Numeric    :", numeric_features)
print("Categorical:", categorical_features)

Numeric    : ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical: ['priority', 'channel']


For the numeric columns I fill missing values with the **median** (it isn't pulled around by outliers like the long tail in tenure) and then **scale** them so they're on a similar footing. Scaling matters for logistic regression and especially k-NN, which relies on distances.

For the categorical columns I fill missing values with the **most frequent** category and then **one-hot encode**, which turns each category into its own 0/1 column. I set `handle_unknown="ignore"` so a category not seen in training won't crash the model.

`ColumnTransformer` applies the right pipeline to the right columns. Because it's fitted only on training data, nothing leaks from the test set.

In [9]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


Quick sanity check: I fit the preprocessor on the training data only, transform both sets, and confirm there are no missing values left and that the column count makes sense.

In [10]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Train shape after preprocessing:", X_train_prep.shape)
print("Test shape after preprocessing :", X_test_prep.shape)
print("Any NaN left in train?", np.isnan(X_train_prep).any())
print("Any NaN left in test? ", np.isnan(X_test_prep).any())
print()
print("Column names:")
print(list(preprocessor.get_feature_names_out()))

Train shape after preprocessing: (675, 9)
Test shape after preprocessing : (225, 9)
Any NaN left in train? False
Any NaN left in test?  False

Column names:
['num__complexity_score', 'num__team_experience_years', 'num__customer_tenure_months', 'cat__priority_high', 'cat__priority_low', 'cat__priority_medium', 'cat__channel_email', 'cat__channel_phone', 'cat__channel_web']
